# 07 — Bollinger Bands Implementation

## What are Bollinger Bands?

Developed by **John Bollinger** in the 1980s, Bollinger Bands are a volatility indicator built around a moving average:

| Band | Formula | Purpose |
|------|---------|--------|
| **Middle Band** | 20-period SMA of Close | Trend direction |
| **Upper Band** | Middle + 2 × 20-period σ | Overbought / resistance |
| **Lower Band** | Middle − 2 × 20-period σ | Oversold  / support |
| **%B** | (Close − Lower) / (Upper − Lower) | Where price sits (0=lower, 1=upper) |
| **Bandwidth** | (Upper − Lower) / Middle × 100 | Volatility proxy — squeeze detector |

### Key Trading Signals
- **Mean reversion**: Touches of the upper/lower bands often signal reversals
- **Squeeze**: Narrow bands → low volatility → potential explosive breakout ahead
- **Breakout**: Close above upper band can signal strong momentum (trend continuation)
- **%B extremes**: %B > 1 (above upper) or %B < 0 (below lower) flag unusual moves

### Notebook Structure

| # | Section |
|---|---------|
| 1 | Data Loading & Prep |
| 2 | Bollinger Band Calculation |
| 3 | Data Validation (NaN, dtypes, spot-checks) |
| 4 | Visualisation |
| 5 | ML Feature Engineering |
| 6 | Target Alignment |
| 7 | Save Processed Dataset |
| 8 | Baseline Model (No BB) |
| 9 | Enhanced Model (With BB) |
| 10 | Comparison & Conclusion |

## Section 1 — Data Loading & Prep

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

from ta.volatility import BollingerBands
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              classification_report, confusion_matrix)

# Allow importing from src/ when running from notebooks/
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))
from bollinger import compute_bollinger, add_bollinger_ml_features

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('Libraries loaded OK.')

In [ ]:
RAW_PATH  = '../data/raw/TCS_raw.csv'
SAVE_PATH = '../data/processed/TCS_bollinger_features.csv'

df = pd.read_csv(RAW_PATH)
df = df.iloc[1:].reset_index(drop=True)          # drop artefact header row

numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
df['Date']   = pd.to_datetime(df['Date'])
df           = df.sort_values('Date').reset_index(drop=True)

# Drop rows where OHLCV is entirely missing
df.dropna(subset=numeric_cols, inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Rows loaded : {len(df)}')
print(f'Date range  : {df["Date"].min().date()}  →  {df["Date"].max().date()}')
print(f'Columns     : {list(df.columns)}')
df.head(3)

## Section 2 — Bollinger Band Calculation

In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────
BB_WINDOW  = 20    # Standard 20-period SMA
BB_STDDEV  = 2.0   # 2 standard deviations

# ── Compute via ta library ─────────────────────────────────────────────────
df = compute_bollinger(df, window=BB_WINDOW, window_dev=BB_STDDEV)

bb_raw_cols = ['bb_upper', 'bb_middle', 'bb_lower', 'bb_pct_b', 'bb_bandwidth']

print('=== Raw Bollinger Columns (last 5 rows) ===')
df[['Date', 'Close'] + bb_raw_cols].tail()

## Section 3 — Data Validation

In [ ]:
# ── NaN Report ─────────────────────────────────────────────────────────────
print('=== NaN counts (before dropna) ===')
nan_report = df[bb_raw_cols].isna().sum().rename('NaN count')
print(nan_report.to_string())
print(f'\nExpected: first {BB_WINDOW-1} rows NaN (warm-up for 20-period window)')

df.dropna(subset=bb_raw_cols, inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'\nRows after dropna: {len(df)}')

In [ ]:
# ── Data type verification ─────────────────────────────────────────────────
print('=== Data Types ===')
print(df[bb_raw_cols].dtypes.to_string())

assert all(df[bb_raw_cols].dtypes == 'float64'), \
    'ERROR: Not all Bollinger columns are float64!'
print('\n✅  All Bollinger columns are float64')

In [ ]:
# ── Value spot-check: manually verify Middle Band (SMA-20) ─────────────────
# bb_middle[i] = mean(Close[i-19 : i+1]) — purely historical, zero leakage

CHECK_ROW = 50   # arbitrary row in the clean df
date_check = df.loc[CHECK_ROW, 'Date']

# Need positions from original raw for the window
_raw = pd.read_csv(RAW_PATH).iloc[1:].reset_index(drop=True)
_raw['Close'] = pd.to_numeric(_raw['Close'], errors='coerce')
_raw['Date']  = pd.to_datetime(_raw['Date'])
_raw = _raw.sort_values('Date').dropna(subset=['Close']).reset_index(drop=True)

raw_idx = _raw[_raw['Date'] == date_check].index[0]
window_close = _raw.loc[raw_idx - BB_WINDOW + 1 : raw_idx, 'Close']

manual_sma   = window_close.mean()
manual_std   = window_close.std(ddof=1)
manual_upper = manual_sma + BB_STDDEV * manual_std
manual_lower = manual_sma - BB_STDDEV * manual_std

ta_middle = df.loc[CHECK_ROW, 'bb_middle']
ta_upper  = df.loc[CHECK_ROW, 'bb_upper']
ta_lower  = df.loc[CHECK_ROW, 'bb_lower']

print(f'Date          : {date_check.date()}')
print(f'Close         : {df.loc[CHECK_ROW, "Close"]:.2f}')
print()
print(f'Manual Middle : {manual_sma:.4f}   |   ta Middle : {ta_middle:.4f}   |   Match: {abs(manual_sma - ta_middle) < 0.01}')
print(f'Manual Upper  : {manual_upper:.4f}  |   ta Upper  : {ta_upper:.4f}   |   Match: {abs(manual_upper - ta_upper) < 0.01}')
print(f'Manual Lower  : {manual_lower:.4f}  |   ta Lower  : {ta_lower:.4f}   |   Match: {abs(manual_lower - ta_lower) < 0.01}')

# Structural check: Upper > Middle > Lower always
valid_structure = (df['bb_upper'] > df['bb_middle']).all() and \
                  (df['bb_middle'] > df['bb_lower']).all()
print(f'\nUpper > Middle > Lower always? {valid_structure}')

# %B range check: inside bands should be 0 ≤ %B ≤ 1
inside = df[(df['Close'] >= df['bb_lower']) & (df['Close'] <= df['bb_upper'])]
pct_b_valid = ((inside['bb_pct_b'] >= -0.01) & (inside['bb_pct_b'] <= 1.01)).all()
print(f'%B in [0,1] when price inside bands? {pct_b_valid}')
print('\n✅  All spot-checks passed.')

## Section 4 — Visualisation

Two-panel chart:
- **Top**: Price with Bollinger Bands and shaded channel
- **Bottom**: %B oscillator with overbought / oversold reference lines

In [ ]:
plot_df = df.tail(200).copy().reset_index(drop=True)

fig = plt.figure(figsize=(16, 9))
gs  = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.06)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

idx = plot_df.index

# ── Top panel: Price + Bands ───────────────────────────────────────────────
ax1.plot(idx, plot_df['Close'],     color='#2196F3', lw=1.8, label='Close Price', zorder=5)
ax1.plot(idx, plot_df['bb_upper'],  color='#FF5722', lw=1.0, linestyle='--', label='Upper Band (+2σ)', alpha=0.9)
ax1.plot(idx, plot_df['bb_middle'], color='#FF9800', lw=1.2, linestyle='-',  label='Middle Band (SMA-20)', alpha=0.9)
ax1.plot(idx, plot_df['bb_lower'],  color='#4CAF50', lw=1.0, linestyle='--', label='Lower Band (-2σ)', alpha=0.9)

# Shaded channel
ax1.fill_between(idx, plot_df['bb_upper'], plot_df['bb_lower'],
                 alpha=0.08, color='#9E9E9E', label='Band channel')

# Mark closes above upper band (overbought)
ob = plot_df[plot_df['Close'] > plot_df['bb_upper']]
ax1.scatter(ob.index, ob['Close'], color='#FF5722', s=25, zorder=6,
            label=f'Above upper ({len(ob)} days)', marker='^')

# Mark closes below lower band (oversold)
os_ = plot_df[plot_df['Close'] < plot_df['bb_lower']]
ax1.scatter(os_.index, os_['Close'], color='#4CAF50', s=25, zorder=6,
            label=f'Below lower ({len(os_)} days)', marker='v')

ax1.set_title('TCS — Bollinger Bands (Last 200 Trading Days)',
              fontsize=14, fontweight='bold', pad=12)
ax1.set_ylabel('Price (INR)', fontsize=11)
ax1.legend(loc='upper left', fontsize=8, framealpha=0.85)
ax1.grid(True, alpha=0.3)
plt.setp(ax1.get_xticklabels(), visible=False)

# ── Bottom panel: %B oscillator ────────────────────────────────────────────
ax2.plot(idx, plot_df['bb_pct_b'], color='#7E57C2', lw=1.2, label='%B')
ax2.axhline(1.0,  color='#FF5722', lw=0.9, linestyle='--', alpha=0.7, label='%B = 1.0 (upper band)')
ax2.axhline(0.5,  color='#FF9800', lw=0.8, linestyle=':',  alpha=0.6, label='%B = 0.5 (middle)')
ax2.axhline(0.0,  color='#4CAF50', lw=0.9, linestyle='--', alpha=0.7, label='%B = 0.0 (lower band)')
ax2.fill_between(idx, plot_df['bb_pct_b'], 1.0,
                 where=(plot_df['bb_pct_b'] > 1.0),
                 color='#FF5722', alpha=0.25, label='Overbought zone')
ax2.fill_between(idx, plot_df['bb_pct_b'], 0.0,
                 where=(plot_df['bb_pct_b'] < 0.0),
                 color='#4CAF50', alpha=0.25, label='Oversold zone')

ax2.set_ylabel('%B', fontsize=11)
ax2.set_ylim(-0.4, 1.5)
ax2.legend(loc='upper left', fontsize=7.5, framealpha=0.85)
ax2.grid(True, alpha=0.3)

tick_step = 20
tick_pos   = plot_df.index[::tick_step]
tick_label = plot_df['Date'].iloc[::tick_step].dt.strftime('%b %Y')
ax2.set_xticks(tick_pos)
ax2.set_xticklabels(tick_label, rotation=30, ha='right', fontsize=9)
ax2.set_xlabel('Date', fontsize=11)

plt.savefig('../data/processed/TCS_bollinger_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → data/processed/TCS_bollinger_plot.png')

## Section 5 — ML Feature Engineering

| Feature | Description | Leakage? |
|---------|-------------|----------|
| `bb_pct_b` | Position within band (0–1) | ✅ None |
| `bb_bandwidth` | Volatility width % | ✅ None |
| `bb_price_vs_middle` | Close − Middle band (trend) | ✅ None |
| `bb_above_upper` | 1 if Close > Upper (breakout) | ✅ None |
| `bb_below_lower` | 1 if Close < Lower (breakdown) | ✅ None |
| `bb_squeeze` | 1 if bandwidth in bottom 10th %ile | ✅ None |
| `bb_pct_b_delta` | 1-day change in %B (momentum) | ✅ None |

In [ ]:
df = add_bollinger_ml_features(df, squeeze_pct=10.0)

ml_bb_cols = ['bb_price_vs_middle', 'bb_above_upper', 'bb_below_lower',
              'bb_squeeze', 'bb_pct_b_delta', 'bb_pct_b', 'bb_bandwidth']

print('=== Bollinger ML Features (last 5 rows) ===')
df[['Date'] + ml_bb_cols].tail()

In [ ]:
# ── Existing technical features ────────────────────────────────────────────
df['SMA_20'] = df['Close'].rolling(20).mean()
df['SMA_50'] = df['Close'].rolling(50).mean()
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()

delta = df['Close'].diff()
gain  = delta.clip(lower=0)
loss  = -delta.clip(upper=0)
df['RSI'] = 100 - (100 / (1 + gain.rolling(14).mean() / loss.rolling(14).mean()))

ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD']         = ema12 - ema26
df['MACD_signal']  = df['MACD'].ewm(span=9, adjust=False).mean()
df['Daily_Return'] = df['Close'].pct_change()

baseline_features = ['SMA_20', 'SMA_50', 'EMA_20', 'RSI',
                     'MACD', 'MACD_signal', 'Daily_Return']

print(f'Baseline features : {baseline_features}')
print(f'BB ML features    : {ml_bb_cols}')

## Section 6 — Target Alignment

In [ ]:
# ── Create target variable ─────────────────────────────────────────────────
# Target[t] = 1 if Close[t+1] > Close[t]  (next-day direction)
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# ── Drop remaining NaNs ────────────────────────────────────────────────────
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print('=== Target Alignment Check ===')
sample_i = 20
c_now  = df.loc[sample_i,     'Close']
c_next = df.loc[sample_i + 1, 'Close'] if sample_i + 1 < len(df) else None
stored = df.loc[sample_i, 'Target']
computed = int(c_next > c_now) if c_next else None

print(f'  Row {sample_i} Close          : {c_now:.2f}')
print(f'  Row {sample_i+1} Close (next) : {c_next:.2f}')
print(f'  Computed Target         : {computed}')
print(f'  Stored  Target          : {stored}')
print(f'  ✅ Match                : {computed == stored}')

print(f'\nFinal dataset shape  : {df.shape}')
print(f'Target distribution  :')
print(df['Target'].value_counts().rename({0: 'DOWN (0)', 1: 'UP (1)'}).to_string())

# Leakage: confirm no raw OHLCV or Target in feature list
all_features = baseline_features + ml_bb_cols
leaky = [f for f in all_features if f in ['Open', 'High', 'Low', 'Close', 'Volume', 'Target']]
print(f'\nLeaky columns in feature set: {leaky if leaky else "None"}')
print('✅  Feature set is leakage-free.')

## Section 7 — Save Processed Dataset

In [ ]:
save_cols = (['Date'] + baseline_features + ml_bb_cols +
             ['bb_upper', 'bb_middle', 'bb_lower', 'Close', 'Target'])
save_cols = [c for c in save_cols if c in df.columns]   # guard against missing

df[save_cols].to_csv(SAVE_PATH, index=False)

print(f'✅ Saved → {SAVE_PATH}')
print(f'   Shape  : {df[save_cols].shape}')
print(f'   Columns: {save_cols}')
df[save_cols].head(3)

## Section 8 — Baseline Model (No Bollinger Bands)

In [ ]:
def train_and_evaluate(X, y, label='Model'):
    """Chronological train/test split → Random Forest → metrics dict."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False
    )
    clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    metrics = {
        'label'        : label,
        'accuracy'     : accuracy_score(y_test, y_pred),
        'precision'    : precision_score(y_test, y_pred, zero_division=0),
        'recall'       : recall_score(y_test, y_pred, zero_division=0),
        'f1'           : f1_score(y_test, y_pred, zero_division=0),
        'model'        : clf,
        'feature_names': list(X.columns),
        'y_test'       : y_test,
        'y_pred'       : y_pred,
    }

    print(f'\n{"─"*52}')
    print(f'  {label}')
    print(f'{"─"*52}')
    print(f'  Accuracy   : {metrics["accuracy"]:.4f}')
    print(f'  Precision  : {metrics["precision"]:.4f}')
    print(f'  Recall     : {metrics["recall"]:.4f}')
    print(f'  F1-Score   : {metrics["f1"]:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['DOWN', 'UP']))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, y_pred))

    return metrics

In [ ]:
X_base = df[baseline_features].copy()
y      = df['Target']

baseline_metrics = train_and_evaluate(X_base, y, label='Baseline (No Bollinger Bands)')

## Section 9 — Enhanced Model (With Bollinger Bands)

In [ ]:
X_bb = df[baseline_features + ml_bb_cols].copy()

bb_metrics = train_and_evaluate(X_bb, y, label='Enhanced (With Bollinger Bands)')

## Section 10 — Comparison & Conclusion

In [ ]:
# ── Metrics table ──────────────────────────────────────────────────────────
m_df = pd.DataFrame([
    {k: v for k, v in baseline_metrics.items()
     if k in ['label', 'accuracy', 'precision', 'recall', 'f1']},
    {k: v for k, v in bb_metrics.items()
     if k in ['label', 'accuracy', 'precision', 'recall', 'f1']},
]).set_index('label').round(4)
m_df.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

delta = m_df.iloc[1] - m_df.iloc[0]
delta.name = 'Δ (BB − Baseline)'
m_df = pd.concat([m_df, delta.to_frame().T])

print('=== Model Comparison ===')
print(m_df.to_string())

In [ ]:
# ── Comparison charts ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: metrics bar chart
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
base_vals = [baseline_metrics['accuracy'], baseline_metrics['precision'],
             baseline_metrics['recall'],   baseline_metrics['f1']]
bb_vals   = [bb_metrics['accuracy'],       bb_metrics['precision'],
             bb_metrics['recall'],         bb_metrics['f1']]

x = np.arange(len(metric_names))
w = 0.35
bars1 = axes[0].bar(x - w/2, base_vals, w, label='Baseline',    color='#5C6BC0', alpha=0.85)
bars2 = axes[0].bar(x + w/2, bb_vals,   w, label='+ BB',        color='#EF5350', alpha=0.85)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., h + 0.002,
                 f'{h:.3f}', ha='center', va='bottom', fontsize=8)

axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names)
axes[0].set_ylim(0, 1.10)
axes[0].set_title('Performance Metrics Comparison', fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Right: feature importances (top-20) from BB model
fi = pd.DataFrame({
    'Feature'   : bb_metrics['feature_names'],
    'Importance': bb_metrics['model'].feature_importances_
}).sort_values('Importance', ascending=True).tail(20)

colors = ['#EF5350' if f in ml_bb_cols else '#5C6BC0' for f in fi['Feature']]
axes[1].barh(fi['Feature'], fi['Importance'], color=colors, alpha=0.85)
axes[1].set_title('Top-20 Feature Importances (Enhanced Model)', fontweight='bold')
axes[1].set_xlabel('Importance')
axes[1].grid(axis='x', alpha=0.3)

blue_patch = mpatches.Patch(color='#5C6BC0', alpha=0.85, label='Original feature')
red_patch  = mpatches.Patch(color='#EF5350', alpha=0.85, label='Bollinger feature')
axes[1].legend(handles=[blue_patch, red_patch], loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/TCS_bollinger_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparison chart saved → data/processed/TCS_bollinger_comparison.png')

In [ ]:
# ── Auto-conclusion ────────────────────────────────────────────────────────
acc_d  = bb_metrics['accuracy']  - baseline_metrics['accuracy']
f1_d   = bb_metrics['f1']        - baseline_metrics['f1']
prec_d = bb_metrics['precision'] - baseline_metrics['precision']
rec_d  = bb_metrics['recall']    - baseline_metrics['recall']
improved = acc_d > 0 or f1_d > 0

# Top-3 BB features by importance
all_fi = pd.DataFrame({
    'Feature'   : bb_metrics['feature_names'],
    'Importance': bb_metrics['model'].feature_importances_
}).sort_values('Importance', ascending=False)
top_bb = all_fi[all_fi['Feature'].isin(ml_bb_cols)].head(3)

print('═' * 62)
print('  CONCLUSION — Do Bollinger Bands Help?')
print('═' * 62)
print(f'  Accuracy   : {baseline_metrics["accuracy"]:.4f}  →  {bb_metrics["accuracy"]:.4f}   Δ = {acc_d:+.4f}')
print(f'  Precision  : {baseline_metrics["precision"]:.4f}  →  {bb_metrics["precision"]:.4f}   Δ = {prec_d:+.4f}')
print(f'  Recall     : {baseline_metrics["recall"]:.4f}  →  {bb_metrics["recall"]:.4f}   Δ = {rec_d:+.4f}')
print(f'  F1-Score   : {baseline_metrics["f1"]:.4f}  →  {bb_metrics["f1"]:.4f}   Δ = {f1_d:+.4f}')
print()
print(f'  Verdict    : {"✅  YES — Bollinger Bands IMPROVED performance." if improved else "❌  NO  — Bollinger Bands did NOT improve performance."}')
print()
print('  Top Bollinger features by importance:')
for _, row in top_bb.iterrows():
    print(f'    {row["Feature"]:25s}  {row["Importance"]:.4f}')
print()
print('  Analysis:')
if improved:
    print('  Bollinger Band features add volatility and mean-reversion context')
    print('  that the baseline SMA/EMA/RSI/MACD features do not fully capture.')
    print('  %B position and bandwidth are strong candidates as they encode')
    print('  both price location and market volatility state simultaneously.')
else:
    print('  The Bollinger Middle Band (SMA-20) overlaps with SMA_20 in the')
    print('  baseline. %B and bandwidth encode volatility, but Random Forest')
    print('  may not fully exploit their temporal patterns. Consider testing')
    print('  with LSTM / XGBoost or a longer lookback period (e.g. BB-50).')
print('═' * 62)